<a href="https://colab.research.google.com/github/dylan-patel108/dylanpatel-DataScience-GenAI-Submissions/blob/main/7_02_Machine_Learning_Modelling_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [1]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [3]:
# Separate predictors and target
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# Check shapes
print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (50000, 15)
Shape of y: (50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [5]:
from sklearn.model_selection import train_test_split

# Split into training (80%) and test (20%) sets preserving class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Check the shapes of the splits
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (40000, 15)
X_test shape: (10000, 15)
y_train shape: (40000,)
y_test shape: (10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [6]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit on training data ONLY and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the fitted scaler
X_test_scaled = scaler.transform(X_test)

# Print summary statistics of one feature before and after scaling to verify
print("Original X_train mean for 'age':", X_train['age'].mean())
print("Original X_train std for 'age':", X_train['age'].std())
print("Scaled X_train mean for first feature:", X_train_scaled[:, 0].mean())
print("Scaled X_train std for first feature:", X_train_scaled[:, 0].std())

Original X_train mean for 'age': 34.91855
Original X_train std for 'age': 11.11856584547466
Scaled X_train mean for first feature: -3.0055957722652236e-16
Scaled X_train std for first feature: 1.0


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Initialize and train Logistic Regression (on scaled data)
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
print("Logistic Regression trained successfully.")

# 2. Initialize and train Random Forest (on unscaled data)
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100)
rf_clf.fit(X_train, y_train)
print("Random Forest trained successfully.")

# 3. Initialize and train Gradient Boosting Classifier (on unscaled data)
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)
print("Gradient Boosting Decision Tree trained successfully.")

Logistic Regression trained successfully.
Random Forest trained successfully.
Gradient Boosting Decision Tree trained successfully.


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate_model(model, X_train_data, X_test_data, y_train, y_test, name):
    # Predictions
    train_preds = model.predict(X_train_data)
    test_preds = model.predict(X_test_data)

    # Metrics calculation
    metrics = {
        'Model': name,
        'Train Accuracy': accuracy_score(y_train, train_preds),
        'Test Accuracy': accuracy_score(y_test, test_preds),
        'Train Macro Precision': precision_score(y_train, train_preds, average='macro'),
        'Test Macro Precision': precision_score(y_test, test_preds, average='macro'),
        'Train Macro Recall': recall_score(y_train, train_preds, average='macro'),
        'Test Macro Recall': recall_score(y_test, test_preds, average='macro'),
        'Train Macro F1': f1_score(y_train, train_preds, average='macro'),
        'Test Macro F1': f1_score(y_test, test_preds, average='macro')
    }
    return metrics

# Evaluate all three models
log_reg_metrics = evaluate_model(log_reg, X_train_scaled, X_test_scaled, y_train, y_test, "Logistic Regression")
rf_metrics = evaluate_model(rf_clf, X_train, X_test, y_train, y_test, "Random Forest")
gb_metrics = evaluate_model(gb_clf, X_train, X_test, y_train, y_test, "Gradient Boosting")

# Present findings in a structured DataFrame
results_df = pd.DataFrame([log_reg_metrics, rf_metrics, gb_metrics])
results_df.set_index('Model', inplace=True)
print(results_df.round(4).to_string())

                     Train Accuracy  Test Accuracy  Train Macro Precision  Test Macro Precision  Train Macro Recall  Test Macro Recall  Train Macro F1  Test Macro F1
Model                                                                                                                                                                
Logistic Regression           0.839         0.8374                 0.8385                0.8363              0.8356             0.8345          0.8367         0.8353
Random Forest                 1.000         0.8844                 1.0000                0.8837              1.0000             0.8824          1.0000         0.8830
Gradient Boosting             0.892         0.8870                 0.8923                0.8874              0.8893             0.8839          0.8905         0.8854


## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?